# Particle size vs. oxygen content (core–shell model)

A spherical metal particle of diameter *D* carries a uniform MO₂ oxide shell of thickness *t*.

- V_core = 4π(r − t)³/3, V_shell = 4πr³/3 − V_core (r = D/2)
- O (wt.%) = 100 · ρ_ox·V_shell·f_O / (ρ_M·V_core + ρ_ox·V_shell), f_O = 2M_O / (M_M + 2M_O)
- SSA (m²/g) = 6 / (ρ_M · D[µm])

**Correction (2026-11):** earlier versions used ρ_M·V_particle (the whole sphere) for the metal mass, which counts the shell volume twice and underestimates oxygen by about 6t/D (≈6 % at 1 µm / 10 nm, ~50 % at 0.2 µm / 50 nm; <1 % above 10 µm). The corrected value is used below; the old value is kept in the CSV (`*_legacy` columns) for comparison.

Interactive version: https://jonglee69.github.io/oxygen_code/


In [ ]:
import csv
import datetime

import matplotlib.pyplot as plt
import numpy as np

print("Particle size vs. oxygen content calculator - developed by J.H. Lee")

# Densities (g/cm^3) and atomic masses (g/mol)
MATERIALS = {
    #      metal density, oxide (MO2) density, metal atomic mass
    "Ti": (4.506, 4.23, 47.867),   # TiO2 rutile
    "Zr": (6.52, 5.68, 91.224),    # ZrO2 monoclinic
    "Hf": (13.31, 9.68, 178.49),   # HfO2 monoclinic
}
M_O = 15.9994


def oxygen_wt(diameter_um, oxide_nm, metal="Ti", legacy=False):
    """Oxygen content (wt.%) of a sphere of diameter D (um) with an MO2 shell t (nm) thick.

    legacy=True reproduces the earlier notebook, which took the metal mass over the
    whole sphere (shell counted twice).
    """
    rho_m, rho_ox, m_metal = MATERIALS[metal]
    f_o = 2 * M_O / (m_metal + 2 * M_O)          # oxygen mass fraction of MO2
    r = 0.5 * diameter_um                          # um
    t = min(oxide_nm / 1000.0, r)                  # um; t >= r means fully oxidised
    v_particle = (4 * np.pi / 3) * r ** 3
    v_core = (4 * np.pi / 3) * (r - t) ** 3
    v_shell = v_particle - v_core
    mass_metal = rho_m * (v_particle if legacy else v_core)
    mass_oxide = rho_ox * v_shell
    return 100 * mass_oxide * f_o / (mass_metal + mass_oxide)


def ssa(diameter_um, metal="Ti"):
    """Specific surface area (m^2/g) of a dense sphere, 6 / (rho * D)."""
    return 6 / (MATERIALS[metal][0] * diameter_um)


# ---- settings ---------------------------------------------------------------
METAL = "Ti"
THICKNESSES_NM = [5, 10, 20, 50]
DIAMETERS_UM = np.arange(1, 2000) * 0.1          # 0.1 ... 199.9 um

# ---- table + CSV ------------------------------------------------------------
stamp = datetime.datetime.now().strftime("%m%d%H%M%S")
data_filename = f"data_{METAL}_{stamp}.csv"
curves = {t: [oxygen_wt(d, t, METAL) for d in DIAMETERS_UM] for t in THICKNESSES_NM}
curves_legacy = {t: [oxygen_wt(d, t, METAL, legacy=True) for d in DIAMETERS_UM] for t in THICKNESSES_NM}
ssa_values = [ssa(d, METAL) for d in DIAMETERS_UM]

with open(data_filename, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(
        ["Diameter (um)", "SSA (m2/g)"]
        + [f"O wt% ({t} nm)" for t in THICKNESSES_NM]
        + [f"O wt% ({t} nm)_legacy" for t in THICKNESSES_NM]
    )
    for i, d in enumerate(DIAMETERS_UM):
        writer.writerow(
            [round(d, 1), ssa_values[i]]
            + [curves[t][i] for t in THICKNESSES_NM]
            + [curves_legacy[t][i] for t in THICKNESSES_NM]
        )
print(f"Data saved as {data_filename}")

for d, t in [(1, 10), (10, 10), (45, 5)]:
    print(f"D = {d:>3} um, t = {t:>2} nm : O = {oxygen_wt(d, t, METAL):.4f} wt.% "
          f"(legacy {oxygen_wt(d, t, METAL, legacy=True):.4f})")

# ---- plot -------------------------------------------------------------------
fig, ax1 = plt.subplots(figsize=(7, 5))
ax2 = ax1.twinx()
for ax in (ax1, ax2):
    ax.set_xscale("log")
    ax.set_yscale("log")
ax1.set_xlim(0.1, 200)
ax1.set_ylim(0.01, 100)
ax2.set_ylim(0.002, 20)
ax1.grid(which="both", alpha=0.4)

ax1.set_xlabel(f"{METAL} particle diameter (µm)")
ax1.set_ylabel("Specific surface area (m²/g)")
ax2.set_ylabel("Oxygen (wt.%)")
ax1.set_title(f"Oxide layer thickness: {THICKNESSES_NM[0]}–{THICKNESSES_NM[-1]} nm")

for t, color, marker in zip(THICKNESSES_NM, ["black", "gray", "blue", "purple"], ["^", "s", "o", "d"]):
    ax2.plot(DIAMETERS_UM, curves[t], color=color, marker=marker, markevery=0.08, label=f"{METAL} ({t} nm)")
ax1.plot(DIAMETERS_UM, ssa_values, color="blue", linestyle="--", label="SSA")
ax1.legend(loc="center right")
ax2.legend(loc="upper right")
plt.tight_layout()
plt.show()
